In [2]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
import itertools
import io
import numpy as np
import json
import re
import zipfile
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

In [3]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="siddiqiya/ar-quran-hadith14books-MSA", 
    repo_type="dataset", local_dir="./ar-quran-hadith14books-MSA", allow_patterns="*/*.parquet")

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 24 files: 100%|██████████| 24/24 [00:09<00:00,  2.61it/s]


'/home/ubuntu/ar-quran-hadith14books-MSA'

In [4]:
files = glob('ar-quran-hadith14books-MSA/*/*.parquet')
len(files)

24

In [5]:
df = pd.read_parquet(files[0])
df

,audio_path,sentence,audio
0,concat_improve_whisper_in_quran_gain_ffmpeg_pa...,فينقلبوا الربا أضعافا وسنجزي وكأين من نبي فتنق...,{'bytes': b'ID3\x04\x00\x00\x00\x00\x00#TSSE\x...
1,concat_improve_whisper_in_quran_gain_ffmpeg_pa...,قتلنا هاهنا تولوا غزى ماتوا قتلوا متم لإلى فظا...,{'bytes': b'ID3\x04\x00\x00\x00\x00\x00#TSSE\x...
2,concat_improve_whisper_in_quran_gain_ffmpeg_pa...,الذين فادرءوا أنما يجتبي بالبينات الغرور الغرو...,{'bytes': b'ID3\x04\x00\x00\x00\x00\x00#TSSE\x...
3,concat_improve_whisper_in_quran_gain_ffmpeg_pa...,فادفعوا وللنساء الأنثيين النصف يوصي أو لهن ترك...,{'bytes': b'ID3\x04\x00\x00\x00\x00\x00#TSSE\x...
4,concat_improve_whisper_in_quran_gain_ffmpeg_pa...,آباؤكم حرمت الأخ اللاتي دخلتم فمن العنت سنن أو...,{'bytes': b'ID3\x04\x00\x00\x00\x00\x00#TSSE\x...
...,...,...,...
195,concat_improve_whisper_in_quran_gain_ffmpeg_pa...,ربهم عقباها تجلى لشتى بالحسنى فسنيسره للعسرى ت...,{'bytes': b'ID3\x04\x00\x00\x00\x00\x00#TSSE\x...
196,concat_improve_whisper_in_quran_gain_ffmpeg_pa...,ضالا بنعمة ورفعنا سينين سافلين اقرأ وربك الأكر...,{'bytes': b'ID3\x04\x00\x00\x00\x00\x00#TSSE\x...
197,concat_improve_whisper_in_quran_gain_ffmpeg_pa...,فليدع ناديه واسجد هي شر أشتاتا شرا شرا يره قدح...,{'bytes': b'ID3\x04\x00\x00\x00\x00\x00#TSSE\x...
198,concat_improve_whisper_in_quran_gain_ffmpeg_pa...,علم وما أدراك ما هيه اليقين لمزة لينبذن الحطمة...,{'bytes': b'ID3\x04\x00\x00\x00\x00\x00#TSSE\x...


In [6]:
def loop(files):

    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1'
    
    files, _ = files

    data = []
    for f in files:
        base = f.split('/')[0] + '_audio'
        f_new = f.replace('/', '-').replace('.parquet', '')
        os.makedirs(base, exist_ok=True)
        df = pd.read_parquet(f)
        for i in tqdm(range(len(df))):
            t = df['sentence'].iloc[i].strip()
            if len(t) < 2:
                continue
            audio_filename = f'{f_new}_{i}.mp3'
            audio_filename = os.path.join(base, audio_filename)
            b = df['audio'].iloc[i]['bytes']
            audio_np, sr = sf.read(io.BytesIO(b))
            if audio_np.ndim > 1:
                audio_np = audio_np.mean(axis=1)
            if audio_np.shape[0] < 10000:
                continue
            sf.write(audio_filename, audio_np, sr)
            
            data.append({
                'audio_filename': audio_filename,
                'text': t,
                'speaker': f"{base}"
            })
        
    return data

In [7]:
data = multiprocessing(files, loop, cores = 5)

100%|██████████| 1456/1456 [04:12<00:00,  5.77it/s]


In [8]:
len(data)

35824

In [9]:
data[0]

{'audio_filename': 'ar-quran-hadith14books-MSA_audio/ar-quran-hadith14books-MSA-data-improve_asr_in_quran-00000-of-00001_0.mp3',
 'text': 'فينقلبوا الربا أضعافا وسنجزي وكأين من نبي فتنقلبوا بل بل الله أشركوا مثوى عفا تلوون لكيلا الأمر',
 'speaker': 'ar-quran-hadith14books-MSA_audio'}

In [10]:
with open('ar-quran-hadith14books-MSA.json', 'w') as fopen:
    json.dump(data, fopen)

In [11]:
audio_files = [d['audio_filename'] for d in data]

with open('ar-quran-hadith14books-MSA-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [14]:
# !zip -rq ar-quran-hadith14books-MSA_audio.zip ar-quran-hadith14books-MSA_audio

In [15]:
# !hf upload malaysia-ai/Multilingual-TTS ar-quran-hadith14books-MSA_audio.zip --repo-type=dataset

In [17]:
# !zip -rq ar-quran-hadith14books-MSA_audio_neucodec.zip ar-quran-hadith14books-MSA_audio_neucodec

In [19]:
# !hf upload malaysia-ai/Multilingual-TTS ar-quran-hadith14books-MSA_audio_neucodec.zip --repo-type=dataset

In [20]:
import json

with open('ar-quran-hadith14books-MSA.json') as fopen:
    rows = json.load(fopen)

mapping = {}
for i in tqdm(range(len(rows))):
    mapping[rows[i]['audio_filename']] = i
len(mapping)

100%|██████████| 35824/35824 [00:00<00:00, 2946499.59it/s]


35824

In [21]:
import faiss
import os
import numpy as np
from tqdm import tqdm

data = {}
d = 192
index = faiss.IndexFlatL2(d)

centroids = []

def assign(x, threshold=0.1):
    if len(centroids) == 0:
        centroids.append(x)
        index.add(np.array([x], dtype=np.float32))
        return 0
    
    D, I = index.search(np.array([x], dtype=np.float32), 1)
    if D[0][0] > threshold:
        centroids.append(x)
        index.add(np.array([x], dtype=np.float32))
        return len(centroids)-1
    else:
        return I[0][0]
        
for i in tqdm(range(len(rows))):
    index_ = mapping[rows[i]['audio_filename']]
    v_f = f'ar-quran-hadith14books-MSA_embedding/{index_}.npy'
    if not os.path.exists(v_f):
        continue
    try:
        v = np.load(v_f)
        data[rows[i]['audio_filename']] = assign(v)
    except Exception as e:
        pass

100%|██████████| 35824/35824 [00:15<00:00, 2305.32it/s]


In [22]:
for i in range(len(rows)):
    s = data[rows[i]['audio_filename']]
    rows[i]['speaker'] = rows[i]['speaker'] + f'_{s}'

In [23]:
from datasets import Dataset

dataset = Dataset.from_list(rows)
dataset[0]

{'audio_filename': 'ar-quran-hadith14books-MSA_audio/ar-quran-hadith14books-MSA-data-improve_asr_in_quran-00000-of-00001_0.mp3',
 'text': 'فينقلبوا الربا أضعافا وسنجزي وكأين من نبي فتنقلبوا بل بل الله أشركوا مثوى عفا تلوون لكيلا الأمر',
 'speaker': 'ar-quran-hadith14books-MSA_audio_0'}

In [25]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'ar-quran-hadith14books-MSA')

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 34.22ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):  93%|█████████▎| 4.46MB / 4.78MB, 22.3MB/s  
Processing Files (1 / 1): 100%|██████████| 4.78MB / 4.78MB, 13.2MB/s  
Processing Files (1 / 1): 100%|██████████| 4.78MB / 4.78MB, 11.9MB/s  
New Data Upload: 100%|██████████| 4.78MB / 4.78MB, 11.9MB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:00<00:00,  1.23 shards/s]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/cf7d76d62792eb8903c46511a437821d633db9f2', commit_message='Upload dataset', commit_description='', oid='cf7d76d62792eb8903c46511a437821d633db9f2', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)